# A notebook to perform QC on the PIPS slow T observations

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import numpy.ma as ma
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.ticker as ticker
import matplotlib.dates as dates
from mpl_toolkits.axes_grid1 import ImageGrid,make_axes_locatable,host_subplot
#from mpl_toolkits.basemap import Basemap
from datetime import datetime, timedelta
import sys
import os
import pyPIPS.utils as utils
import pyPIPS.thermolib as thermo
import pyPIPS.DSDlib as dsd
#import pyPIPS.disdrometer_module as dis
import pyPIPS.plotmodule as PIPSplot
#import pyPIPS.simulator as sim
import pyPIPS.pips_io as pipsio
import pyPIPS.PIPS as pips
import pyPIPS.parsivel_params as pp
import pyPIPS.parsivel_qc as pqc
import pyPIPS.polarimetric as dualpol
#from pyCRMtools.modules import plotmodule as plotmod
from pyCRMtools.modules import utils as CRMutils
# from pyCRMtools.pycaps import arps_read
# from pyCRMtools.pycaps import pycaps_fields
# from pyCRMtools.pycaps import calvars_radar as radar
import pandas as pd
import xarray as xr
import xskillscore as xs
import glob
import numpy.random as random
from scipy.stats import gamma, uniform
from scipy.stats.mstats import zscore
from scipy.special import gamma as gammafunc
from scipy import ndimage
from metpy.plots import StationPlot
from metpy.calc import wind_components
from metpy.cbook import get_test_data
from metpy.plots import StationPlot
from metpy.plots.wx_symbols import current_weather, sky_cover
from metpy.units import units
from cycler import cycler
import warnings
warnings.simplefilter('ignore')
%matplotlib widget

In [ ]:
# plt.style.use('seaborn-v0_8-bright')

In [ ]:
# Read in the PIPS netcdf files for the case we want

# PIPS_input_base_dir = '/Users/dawson29/Projects/PERiLS/obsdata/2022/PIPS_data/'
# PIPS_output_base_dir = '/Users/dawson29/Projects/PERiLS/obsdata/2022/PIPS_data_for_EOL/'

# PIPS_base_dir = '/Users/dawson29/Projects/PERiLS/obsdata/2023/PIPS_data/'
# PIPS_base_dir = '/Users/dawson29/Dropbox/Projects/PERiLS/obsdata/2023/'
# PIPS_base_dir = '/Users/dawson29/Projects/PERiLS/obsdata/2023/PIPS_data/'
# PIPS_base_dir = '/Users/dawson29/Projects/PERiLS/obsdata/2022/PIPS_data/'
PIPS_base_dir = '/Users/dawson29/Dropbox/Projects/ICECHIP/obsdata/PIPS_data/'
deployment_dirs = glob.glob(PIPS_base_dir + 'IOP*')
deployment_names = [os.path.basename(deployment_dir) for deployment_dir in deployment_dirs]
PIPS_input_dirs = [os.path.join(deployment_dir, 'netcdf') for deployment_dir in deployment_dirs]

# PIPS_output_dir = os.path.join(PIPS_base_dir, deployment_name, 'netcdf_thermoQC_slowtemp_only')


# if not os.path.exists(PIPS_output_dir):
#     os.makedirs(PIPS_output_dir)

PIPS_name = 'PIPS3B'  # Choose which PIPS to analyze
# PIPS_names = ['PIPS1A', 'PIPS1B', 'PIPS2A', 'PIPS2B', 'PIPS3A', 'PIPS3B']
parsivel_interval = 10
intervalstr = '10S'

parsivel_filenames = ['parsivel_combined_{}_{}_{:d}s.nc'.format(deployment_name, PIPS_name, parsivel_interval)
                      for deployment_name in deployment_names]
parsivel_filepaths = [os.path.join(PIPS_input_dir, parsivel_filename)
                      for PIPS_input_dir, parsivel_filename in zip(PIPS_input_dirs, parsivel_filenames)]
# output_parsivel_filepaths = [os.path.join(PIPS_output_dir, parsivel_filename)
#                              for parsivel_filename in parsivel_filenames]
conv_filenames = ['conventional_raw_{}_{}.nc'.format(deployment_name, PIPS_name) for deployment_name in deployment_names]
conv_filepaths = [os.path.join(PIPS_input_dir, conv_filename) for PIPS_input_dir, conv_filename in zip(PIPS_input_dirs, conv_filenames)]
# output_conv_filepaths = [os.path.join(PIPS_output_dir, conv_filename) for conv_filename in conv_filenames]
parsivel_ds_dict = {}
conv_ds_dict = {}
for deployment_name, parsivel_filepath, conv_filepath in zip(deployment_names, parsivel_filepaths, conv_filepaths):
    dict_key = PIPS_name + '_' + deployment_name
    try:
        parsivel_ds_dict[dict_key] = xr.open_dataset(parsivel_filepath)
    except:
        continue
    try:
        conv_ds_dict[dict_key] = xr.open_dataset(conv_filepath)
    except:
        continue

In [ ]:
# Load all slowtemp DataArrays from all deployments and concatenate them together for QC analysis
all_slowtemp_das = []
for dict_key in conv_ds_dict.keys():
    if dict_key.startswith(PIPS_name):
        try:
            slowtemp_da = conv_ds_dict[dict_key]['slowtemp']
            # Drop 'flagged_times' coordinate if it exists to avoid concat errors
            if 'flagged_times' in slowtemp_da.coords:
                slowtemp_da = slowtemp_da.drop_vars('flagged_times')
            all_slowtemp_das.append(slowtemp_da)
        except KeyError:
            continue
all_slowtemp_da = xr.concat(all_slowtemp_das, dim='time')
# Do the same for the fasttemps
all_fasttemp_das = []
for dict_key in conv_ds_dict.keys():
    if dict_key.startswith(PIPS_name):
        try:
            fasttemp_da = conv_ds_dict[dict_key]['fasttemp']
            # Drop 'flagged_times' coordinate if it exists to avoid concat errors
            if 'flagged_times' in fasttemp_da.coords:
                fasttemp_da = fasttemp_da.drop_vars('flagged_times')
            all_fasttemp_das.append(fasttemp_da)
        except KeyError:
            continue
all_fasttemp_da = xr.concat(all_fasttemp_das, dim='time')

In [ ]:
# Make a plot showing pairwise scatter of slowtemp vs fasttemp with equal axes and 1:1 line.
# Also compute linear fit and statistics

# Remove NaN values for statistics
fasttemp_valid = all_fasttemp_da.values[~np.isnan(all_fasttemp_da.values) & ~np.isnan(all_slowtemp_da.values)]
slowtemp_valid = all_slowtemp_da.values[~np.isnan(all_fasttemp_da.values) & ~np.isnan(all_slowtemp_da.values)]

# Calculate statistics
bias = np.mean(slowtemp_valid - fasttemp_valid)
rmse = np.sqrt(np.mean((slowtemp_valid - fasttemp_valid)**2))
corr_coef = np.corrcoef(fasttemp_valid, slowtemp_valid)[0, 1]
r_squared = corr_coef**2

# Linear regression fit
poly_coeffs = np.polyfit(fasttemp_valid, slowtemp_valid, 1)
slope = poly_coeffs[0]
intercept = poly_coeffs[1]

# Plot
plt.figure(figsize=(8, 8))
plt.scatter(all_fasttemp_da, all_slowtemp_da, s=1, alpha=0.5, label='Data')

# Determine common limits and enforce equal ranges
data_min = np.nanmin([all_fasttemp_da.min(), all_slowtemp_da.min()])
data_max = np.nanmax([all_fasttemp_da.max(), all_slowtemp_da.max()])

# Plot 1:1 line
plt.plot([data_min, data_max], [data_min, data_max], 'k--', lw=1.5, label='1:1')

# Plot linear fit line
x_fit = np.array([data_min, data_max])
y_fit = slope * x_fit + intercept
plt.plot(x_fit, y_fit, 'r-', lw=2, label=f'Linear Fit: y={slope:.3f}x+{intercept:.3f}')

plt.xlim(data_min, data_max)
plt.ylim(data_min, data_max)

plt.xlabel('Fast Temperature (°C)', fontsize=12)
plt.ylabel('Slow Temperature (°C)', fontsize=12)
plt.title(f'Pairwise Scatter of Slowtemp vs Fasttemp for {PIPS_name}', fontsize=13)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right', fontsize=10)
plt.axis('equal')

# Add statistics text box
stats_text = f'N = {len(fasttemp_valid):,}\n'
stats_text += f'Bias = {bias:.3f} °C\n'
stats_text += f'RMSE = {rmse:.3f} °C\n'
stats_text += f'r = {corr_coef:.4f}\n'
stats_text += f'R² = {r_squared:.4f}\n'
stats_text += f'Slope = {slope:.4f}\n'
stats_text += f'Intercept = {intercept:.3f} °C'

plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

# Print statistics summary
print(f'\nStatistical Summary for {PIPS_name}:')
print(f'Sample size: {len(fasttemp_valid):,}')
print(f'Bias (Slow - Fast): {bias:.4f} °C')
print(f'RMSE: {rmse:.4f} °C')
print(f'Correlation coefficient (r): {corr_coef:.4f}')
print(f'R-squared: {r_squared:.4f}')
print(f'Linear fit: Slow = {slope:.4f} * Fast + {intercept:.4f}')

In [ ]:
# Detect and remove periods with rapid temperature changes (e.g., gust fronts)
# Also remove observations with large temperature differences between sensors
#
# Strategy:
# 1. Calculate the rate of temperature change using fasttemp (°C per time step)
# 2. Use a rolling window to identify sustained periods of rapid change
# 3. Remove observations where |slowtemp - fasttemp| exceeds a threshold
# 4. Remove these periods before computing bias correction
#
# This is necessary because during rapid temperature changes (like gust front passages),
# the slow and fast sensors differ due to their different response times, not systematic bias.
# Large differences may also indicate sensor malfunctions or contamination.

# Calculate the time derivative of fasttemp (rate of change)
# First, compute time differences in seconds (keep as xarray DataArray)
time_diff = all_fasttemp_da.time.diff('time')
time_diff_seconds = time_diff / np.timedelta64(1, 's')

# Identify deployment boundaries (large time gaps)
# Normal data is at 1s intervals, so gaps > 60s indicate deployment boundaries
max_normal_gap = 60.0  # seconds
deployment_boundary_mask = time_diff_seconds > max_normal_gap

# Compute temperature differences
fasttemp_diff = all_fasttemp_da.diff('time', label='lower')

# Rate of change in °C per second
dT_dt = np.abs(fasttemp_diff / time_diff_seconds)

# Set dT_dt to NaN at deployment boundaries to prevent rolling window from crossing them
dT_dt = dT_dt.where(~deployment_boundary_mask)

# Report boundary detection
n_boundaries = int(deployment_boundary_mask.sum())
print(f"\nDeployment Boundary Detection:")
print(f"  Max normal time gap: {max_normal_gap} seconds")
print(f"  Number of deployment boundaries detected: {n_boundaries}")
if n_boundaries > 0:
    boundary_indices = np.where(deployment_boundary_mask.values)[0]
    print(f"  Boundary locations (time indices): {boundary_indices}")
    for idx in boundary_indices:
        gap_hours = float(time_diff_seconds.values[idx]) / 3600.0
        print(f"    Gap at index {idx}: {gap_hours:.1f} hours")

# Define threshold for "rapid change" - using 0.01 °C/s as threshold
# (equivalent to 0.6 °C/minute or 36 °C/hour, which is very rapid for atmospheric conditions)
rapid_change_threshold = 0.01  # °C/s

# Use a rolling window to identify sustained rapid change periods
# A window of 60 timesteps (~60 seconds at 1s intervals) helps identify true events
window_size = 60
dT_dt_rolling_mean = dT_dt.rolling(time=window_size, center=True, min_periods=1).mean()

# Create a mask: True where temperature is changing rapidly
rapid_change_mask = dT_dt_rolling_mean > rapid_change_threshold

# Since diff reduces array size by 1, align mask with original data
# Extend mask to match original array length (replicate last value)
rapid_change_mask_full = xr.concat([rapid_change_mask,
                                     xr.DataArray([rapid_change_mask.values[-1]],
                                                 coords={'time': [all_fasttemp_da.time.values[-1]]})],
                                    dim='time')

# Additional filter: Remove observations where slowtemp and fasttemp differ by more than threshold
temp_diff_threshold = 100.0  # °C
temp_diff = np.abs(all_slowtemp_da - all_fasttemp_da)
large_diff_mask = temp_diff > temp_diff_threshold

# Combine both masks: remove points that meet EITHER criterion
combined_mask = rapid_change_mask_full | large_diff_mask

# Filter out flagged periods from both temperature arrays
fasttemp_filtered = all_fasttemp_da.where(~combined_mask, drop=True)
slowtemp_filtered = all_slowtemp_da.where(~combined_mask, drop=True)

# Report filtering results
n_total = len(all_fasttemp_da)
n_removed_rapid = rapid_change_mask_full.sum().values
n_removed_diff = large_diff_mask.sum().values
n_removed_total = combined_mask.sum().values
n_remaining = n_total - n_removed_total
pct_removed = (n_removed_total / n_total) * 100

print(f"\nTemperature Filtering Results:")
print(f"  Rapid change threshold: {rapid_change_threshold} °C/s")
print(f"  Temperature difference threshold: {temp_diff_threshold} °C")
print(f"  Rolling window: {window_size} timesteps")
print(f"  Total data points: {n_total:,}")
print(f"  Points removed (rapid change): {n_removed_rapid:,} ({(n_removed_rapid/n_total)*100:.1f}%)")
print(f"  Points removed (large difference): {n_removed_diff:,} ({(n_removed_diff/n_total)*100:.1f}%)")
print(f"  Points removed (total): {n_removed_total:,} ({pct_removed:.1f}%)")
print(f"  Points remaining: {n_remaining:,} ({100-pct_removed:.1f}%)")


In [ ]:
# Create scatter plot comparing filtered vs. unfiltered data

# Remove NaN values for statistics - FILTERED DATA
fasttemp_valid_filt = fasttemp_filtered.values[~np.isnan(fasttemp_filtered.values) & ~np.isnan(slowtemp_filtered.values)]
slowtemp_valid_filt = slowtemp_filtered.values[~np.isnan(fasttemp_filtered.values) & ~np.isnan(slowtemp_filtered.values)]

# Calculate statistics for filtered data
bias_filt = np.mean(slowtemp_valid_filt - fasttemp_valid_filt)
rmse_filt = np.sqrt(np.mean((slowtemp_valid_filt - fasttemp_valid_filt)**2))
corr_coef_filt = np.corrcoef(fasttemp_valid_filt, slowtemp_valid_filt)[0, 1]
r_squared_filt = corr_coef_filt**2

# Linear regression fit for filtered data
poly_coeffs_filt = np.polyfit(fasttemp_valid_filt, slowtemp_valid_filt, 1)
slope_filt = poly_coeffs_filt[0]
intercept_filt = poly_coeffs_filt[1]

# Create side-by-side comparison plots
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# LEFT PLOT: Original unfiltered data (using previous results)
ax = axes[0]
ax.scatter(all_fasttemp_da, all_slowtemp_da, s=1, alpha=0.5, label='Data', c='gray')

data_min = np.nanmin([all_fasttemp_da.min(), all_slowtemp_da.min()])
data_max = np.nanmax([all_fasttemp_da.max(), all_slowtemp_da.max()])

ax.plot([data_min, data_max], [data_min, data_max], 'k--', lw=1.5, label='1:1')

x_fit = np.array([data_min, data_max])
y_fit = slope * x_fit + intercept
ax.plot(x_fit, y_fit, 'r-', lw=2, label=f'Fit: y={slope:.3f}x+{intercept:.3f}')

ax.set_xlim(data_min, data_max)
ax.set_ylim(data_min, data_max)
ax.set_xlabel('Fast Temperature (°C)', fontsize=12)
ax.set_ylabel('Slow Temperature (°C)', fontsize=12)
ax.set_title('Original Data (Unfiltered)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=10)
ax.set_aspect('equal')

stats_text = f'N = {len(fasttemp_valid):,}\n'
stats_text += f'Bias = {bias:.3f} °C\n'
stats_text += f'RMSE = {rmse:.3f} °C\n'
stats_text += f'r = {corr_coef:.4f}\n'
stats_text += f'R² = {r_squared:.4f}\n'
stats_text += f'Slope = {slope:.4f}\n'
stats_text += f'Intercept = {intercept:.3f} °C'

ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# RIGHT PLOT: Filtered data (rapid change periods removed)
ax = axes[1]
ax.scatter(fasttemp_filtered, slowtemp_filtered, s=1, alpha=0.5, label='Data', c='blue')

ax.plot([data_min, data_max], [data_min, data_max], 'k--', lw=1.5, label='1:1')

y_fit_filt = slope_filt * x_fit + intercept_filt
ax.plot(x_fit, y_fit_filt, 'r-', lw=2, label=f'Fit: y={slope_filt:.3f}x+{intercept_filt:.3f}')

ax.set_xlim(data_min, data_max)
ax.set_ylim(data_min, data_max)
ax.set_xlabel('Fast Temperature (°C)', fontsize=12)
ax.set_ylabel('Slow Temperature (°C)', fontsize=12)
ax.set_title('Filtered Data (Rapid Changes Removed)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=10)
ax.set_aspect('equal')

stats_text_filt = f'N = {len(fasttemp_valid_filt):,}\n'
stats_text_filt += f'Bias = {bias_filt:.3f} °C\n'
stats_text_filt += f'RMSE = {rmse_filt:.3f} °C\n'
stats_text_filt += f'r = {corr_coef_filt:.4f}\n'
stats_text_filt += f'R² = {r_squared_filt:.4f}\n'
stats_text_filt += f'Slope = {slope_filt:.4f}\n'
stats_text_filt += f'Intercept = {intercept_filt:.3f} °C'

ax.text(0.02, 0.98, stats_text_filt, transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.show()

# Print comparison summary
print(f"\n{'='*60}")
print("COMPARISON: Original vs. Filtered Data")
print(f"{'='*60}")
print(f"                     Original        Filtered       Change")
print(f"{'-'*60}")
print(f"Sample Size:      {len(fasttemp_valid):>10,}   {len(fasttemp_valid_filt):>10,}   {len(fasttemp_valid_filt)-len(fasttemp_valid):>+10,}")
print(f"Bias (°C):        {bias:>10.4f}   {bias_filt:>10.4f}   {bias_filt-bias:>+10.4f}")
print(f"RMSE (°C):        {rmse:>10.4f}   {rmse_filt:>10.4f}   {rmse_filt-rmse:>+10.4f}")
print(f"Correlation (r):  {corr_coef:>10.4f}   {corr_coef_filt:>10.4f}   {corr_coef_filt-corr_coef:>+10.4f}")
print(f"R²:               {r_squared:>10.4f}   {r_squared_filt:>10.4f}   {r_squared_filt-r_squared:>+10.4f}")
print(f"Slope:            {slope:>10.4f}   {slope_filt:>10.4f}   {slope_filt-slope:>+10.4f}")
print(f"Intercept (°C):   {intercept:>10.4f}   {intercept_filt:>10.4f}   {intercept_filt-intercept:>+10.4f}")
print(f"{'='*60}")
print(f"\nFiltering improved R² by {(r_squared_filt-r_squared)*100:.2f} percentage points")
print(f"Filtering reduced RMSE by {(rmse-rmse_filt):.4f} °C ({((rmse-rmse_filt)/rmse)*100:.1f}% reduction)")

In [ ]:
print('slope_filt: ', slope_filt)
print('intercept_filt: ', intercept_filt)

In [ ]:
# Remove any "corrected" thermodynamic variables that are already in the datasets
vars_to_remove = ['fasttemp_corrected', 'slowtemp_corrected', 'RH_corrected', 'pressure_corrected',
                  'dewpoint_corrected', 'RH_derived_corrected', 'pt_corrected', 'qv_corrected',
                  'rho_corrected']

for dict_key in parsivel_ds_dict.keys():
    if dict_key.startswith(PIPS_name):
        try:
            conv_ds_dict[PIPS_name] = conv_ds_dict[dict_key].drop_vars(vars_to_remove)
        except (AttributeError, ValueError):
            print("Vars already removed")
        try:
            parsivel_ds_dict[PIPS_name] = parsivel_ds_dict[dict_key].drop_vars(vars_to_remove)
        except (AttributeError, ValueError):
            print("Vars already removed")

In [ ]:
# Create copies of the slowtemps for each PIPS which will be filled with corrected data
for dict_key in conv_ds_dict.keys():
    if dict_key.startswith(PIPS_name):
        conv_ds_dict[dict_key]['slowtemp_corrected'] = conv_ds_dict[dict_key]['slowtemp'].copy()

In [ ]:
# Correct the slowtemps using the linear regression formula computed above

all_slowtemp_corrected_das = []
for dict_key in conv_ds_dict.keys():
    if dict_key.startswith(PIPS_name):
        # Your regression gives: slowtemp = slope * fasttemp + intercept
        # To correct slowtemp, invert this relationship:
        conv_ds_dict[dict_key]['slowtemp_corrected'] = (conv_ds_dict[dict_key]['slowtemp'] - intercept_filt) / slope_filt
        conv_ds_dict[dict_key]['slowtemp_corrected'].attrs['slope'] = slope_filt
        conv_ds_dict[dict_key]['slowtemp_corrected'].attrs['intercept'] = intercept_filt

        all_slowtemp_corrected_das.append(conv_ds_dict[dict_key]['slowtemp_corrected'])

all_slowtemp_corrected_da = xr.concat(all_slowtemp_corrected_das, dim='time')

In [ ]:
# Make a plot showing pairwise scatter of corrected slowtemp vs fasttemp with equal axes and 1:1 line.
# Also compute linear fit and statistics

# Remove NaN values for statistics
fasttemp_valid = all_fasttemp_da.values[~np.isnan(all_fasttemp_da.values) & ~np.isnan(all_slowtemp_corrected_da.values)]
slowtemp_corrected_valid = all_slowtemp_corrected_da.values[~np.isnan(all_fasttemp_da.values) & ~np.isnan(all_slowtemp_corrected_da.values)]

# Calculate statistics
bias = np.mean(slowtemp_corrected_valid - fasttemp_valid)
rmse = np.sqrt(np.mean((slowtemp_corrected_valid - fasttemp_valid)**2))
corr_coef = np.corrcoef(fasttemp_valid, slowtemp_corrected_valid)[0, 1]
r_squared = corr_coef**2

# Linear regression fit
poly_coeffs = np.polyfit(fasttemp_valid, slowtemp_corrected_valid, 1)
slope = poly_coeffs[0]
intercept = poly_coeffs[1]

# Plot
plt.figure(figsize=(8, 8))
plt.scatter(all_fasttemp_da, all_slowtemp_corrected_da, s=1, alpha=0.5, label='Data')

# Determine common limits and enforce equal ranges
data_min = np.nanmin([all_fasttemp_da.min(), all_slowtemp_corrected_da.min()])
data_max = np.nanmax([all_fasttemp_da.max(), all_slowtemp_corrected_da.max()])

# Plot 1:1 line
plt.plot([data_min, data_max], [data_min, data_max], 'k--', lw=1.5, label='1:1')

# Plot linear fit line
x_fit = np.array([data_min, data_max])
y_fit = slope * x_fit + intercept
plt.plot(x_fit, y_fit, 'r-', lw=2, label=f'Linear Fit: y={slope:.3f}x+{intercept:.3f}')

plt.xlim(data_min, data_max)
plt.ylim(data_min, data_max)

plt.xlabel('Fast Temperature (°C)', fontsize=12)
plt.ylabel('Slow Temperature (°C)', fontsize=12)
plt.title(f'Pairwise Scatter of Corrected Slowtemp vs Fasttemp for {PIPS_name}', fontsize=13)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right', fontsize=10)
plt.axis('equal')

# Add statistics text box
stats_text = f'N = {len(fasttemp_valid):,}\n'
stats_text += f'Bias = {bias:.3f} °C\n'
stats_text += f'RMSE = {rmse:.3f} °C\n'
stats_text += f'r = {corr_coef:.4f}\n'
stats_text += f'R² = {r_squared:.4f}\n'
stats_text += f'Slope = {slope:.4f}\n'
stats_text += f'Intercept = {intercept:.3f} °C'

plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

# Print statistics summary
print(f'\nStatistical Summary for {PIPS_name}:')
print(f'Sample size: {len(fasttemp_valid):,}')
print(f'Bias (Slow - Fast): {bias:.4f} °C')
print(f'RMSE: {rmse:.4f} °C')
print(f'Correlation coefficient (r): {corr_coef:.4f}')
print(f'R-squared: {r_squared:.4f}')
print(f'Linear fit: Slow = {slope:.4f} * Fast + {intercept:.4f}')

In [ ]:
# Now, we need to recompute the dewpoint and RH_derived using the bias-corrected slowtemp from above
RH_das = []
RH_derived_das = []
RH_derived_corrected_das = []
for dict_key in conv_ds_dict.keys():
    if dict_key.startswith(PIPS_name):

        pressure = conv_ds_dict[dict_key]['pressure']
        slowtemp = conv_ds_dict[dict_key]['slowtemp_corrected']
        fasttemp = conv_ds_dict[dict_key]['fasttemp']
        RH = conv_ds_dict[dict_key]['RH']
        RH_derived = conv_ds_dict[dict_key]['RH_derived']
        RH_das.append(RH)
        RH_derived_das.append(RH_derived)
        dewpoint = thermo.calTdfromRH(pressure * 100., slowtemp + 273.15, RH / 100.) - 273.15
    #     dewpoint.sel(time=slice(time_start, time_stop)).plot(ax=ax, label=f'{PIPS_name}_dewpoint',
    #                                                          ls='None', marker='o', ms=1., alpha=0.5)
        RH_derived_corrected = thermo.calRH(pressure * 100., fasttemp + 273.15, dewpoint + 273.15) * 100.

        conv_ds_dict[dict_key]['dewpoint_corrected'] = conv_ds_dict[dict_key]['dewpoint'].copy()
        conv_ds_dict[dict_key]['dewpoint_corrected'].data = dewpoint

        conv_ds_dict[dict_key]['RH_derived_corrected'] = RH_derived.copy()
        conv_ds_dict[dict_key]['RH_derived_corrected'].data = RH_derived_corrected
        RH_derived_corrected_das.append(conv_ds_dict[dict_key]['RH_derived_corrected'])

# Concatenate the RH DataArrays
all_RH_da = xr.concat(RH_das, dim='time')
all_RH_derived_da = xr.concat(RH_derived_das, dim='time')
all_RH_derived_corrected_da = xr.concat(RH_derived_corrected_das, dim='time')

In [ ]:
# Create histogram plots for RH, RH_derived, and RH_derived_corrected

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Use the concatenated DataArrays from the previous cell
RH_all = all_RH_da.values[~np.isnan(all_RH_da.values)]
RH_derived_all = all_RH_derived_da.values[~np.isnan(all_RH_derived_da.values)]
RH_derived_corrected_all = all_RH_derived_corrected_da.values[~np.isnan(all_RH_derived_corrected_da.values)]

# Define common bin edges for all histograms (0 to 120% in steps of 2%)
# Extended beyond 100% to capture any supersaturated values
bins = np.arange(0, 122, 2)

# Plot histogram for original RH
axes[0].hist(RH_all, bins=bins, color='blue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Relative Humidity (%)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Original RH', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(np.mean(RH_all), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(RH_all):.2f}%')
axes[0].legend()

# Plot histogram for original RH_derived
axes[1].hist(RH_derived_all, bins=bins, color='green', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Relative Humidity (%)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Original RH_derived', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(np.mean(RH_derived_all), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(RH_derived_all):.2f}%')
axes[1].legend()

# Plot histogram for corrected RH_derived
axes[2].hist(RH_derived_corrected_all, bins=bins, color='orange', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Relative Humidity (%)', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].set_title('Corrected RH_derived', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].axvline(np.mean(RH_derived_corrected_all), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(RH_derived_corrected_all):.2f}%')
axes[2].legend()

plt.tight_layout()
plt.show()

# Print summary statistics
print(f'\nRH Statistics for {PIPS_name}:')
print(f'  Original RH: Mean = {np.mean(RH_all):.2f}%, Std = {np.std(RH_all):.2f}%')
print(f'  Original RH_derived: Mean = {np.mean(RH_derived_all):.2f}%, Std = {np.std(RH_derived_all):.2f}%')
print(f'  Corrected RH_derived: Mean = {np.mean(RH_derived_corrected_all):.2f}%, Std = {np.std(RH_derived_corrected_all):.2f}%')